In [1]:
import numpy as np
import math
import rclpy
import threading
import time

from semantic_digital_twin.world import World
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.robots.soft_trunk import SoftTrunk


/home/rinaalo/.virtualenvs/cram-env/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
# Initialize ROS 2 Node
if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node("soft_pcc_test")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

In [3]:
# Create the snake with 3 sections, and 10 visual segments per section
num_sections = 5
world = World()
trunk, kappas, phis = SoftTrunk.build_pcc(world, num_sections=num_sections, segs_per_section=10, total_robot_length=2.0)

In [4]:
# Setup Visualization
tf_pub = TFPublisher(_world=world, node=node)
viz_pub = VizMarkerPublisher(_world=world, node=node)
tf_pub.add_to_world(world)
viz_pub.add_to_world(world)

print(f"Robot '{trunk.name.name}' is ready. Check RViz.")

Robot 'soft_snake' is ready. Check RViz.


In [5]:
try:
    while True:
        print("Animating Snake: Propagating Wave...")
        for t in np.linspace(0, 20, 400):
            
            for s in range(num_sections):
                # The 's * 1.0' creates a delay so section 2 starts bending 
                # slightly after section 1.
                k_val = 1.5 * np.sin(t - s * 1.0) 
                
                # make it spiral by rotating the phis
                p_val = t * 0.5 
                
                # Update the specific DOFs for this section
                world.state[kappas[s].id].position = float(k_val)
                world.state[phis[s].id].position = float(p_val)
            
            # Recompute everything
            world.notify_state_change()
            
            # Update RViz
            tf_pub.notify()
            viz_pub.notify()
            
            time.sleep(0.03)
        
except KeyboardInterrupt:
    print("Animation stopped.")

Animating Snake: Propagating Wave...
Animating Snake: Propagating Wave...
Animation stopped.


In [6]:
node.destroy_node()
rclpy.shutdown()